In [ ]:
!git clone https://github.com/tum-bgd/CVDisaster.git
%cd CVDisaster/geolocalization
!pip install -r requirement.txt
!pip install geopandas

In [ ]:
!wget "https://dataserv.ub.tum.de/index.php/s/m1749324/download" -O m1749324.zip

In [ ]:
!unzip -q m1749324.zip -d /content/CVDisaster/geolocalization/
!mv /content/CVDisaster/geolocalization/m1749324/CVIAN /content/CVDisaster/geolocalization/CVIAN

In [ ]:
!python 0_prepare_splits_and_data.py

In [ ]:
!sed -i "s/batch_size: int = 128/batch_size: int = 8/" /content/CVDisaster/geolocalization/1_train_cvdisaster.py
!sed -i "s/gpu_ids: tuple = (0, 1, 2, 3)/gpu_ids: tuple = (0,)/" /content/CVDisaster/geolocalization/1_train_cvdisaster.py
!sed -i 's#model_path: str = "./cvdisaster"#model_path: str = "/content/drive/MyDrive/cvdisaster"#' /content/CVDisaster/geolocalization/1_train_cvdisaster.py

In [ ]:
!grep -n "batch_size: int\|gpu_ids: tuple\|model_path: str" /content/CVDisaster/geolocalization/1_train_cvdisaster.py

In [ ]:
!pip install gdown -q
!gdown --folder "https://drive.google.com/drive/folders/1PMuUqvDnCb216D8_ZDDJzDD3FxeH5BoA" -O /content/sample4geo_pretrained

In [ ]:
!unzip -q /content/sample4geo_pretrained/pretrained.zip -d /content/sample4geo_pretrained/
!find /content/sample4geo_pretrained -iname "*.pth" | grep -i cvusa

In [ ]:
!cp "/content/sample4geo_pretrained/pretrained/cvusa/convnext_base.fb_in22k_ft_in1k_384/weights_e40_98.6830.pth" /content/CVDisaster/geolocalization/weights_e40_98.6830.pth
!ls -lh /content/CVDisaster/geolocalization/weights_e40_98.6830.pth

In [ ]:
%cd /content/CVDisaster/geolocalization
!python 1_train_cvdisaster.py

In [ ]:
!find /content -iname "weights_e40_98.6830.pth"

In [ ]:
!find /content/drive/MyDrive/cvdisaster -name "*.pth"

/content/drive/MyDrive/cvdisaster/convnext_base.fb_in22k_ft_in1k_384/212347/weights_e1_28.3582.pth
/content/drive/MyDrive/cvdisaster/convnext_base.fb_in22k_ft_in1k_384/212347/weights_e4_31.3433.pth
/content/drive/MyDrive/cvdisaster/convnext_base.fb_in22k_ft_in1k_384/212347/weights_e5_38.4328.pth
/content/drive/MyDrive/cvdisaster/convnext_base.fb_in22k_ft_in1k_384/212347/weights_e6_44.0299.pth
/content/drive/MyDrive/cvdisaster/convnext_base.fb_in22k_ft_in1k_384/212347/weights_e7_53.3582.pth
/content/drive/MyDrive/cvdisaster/convnext_base.fb_in22k_ft_in1k_384/212347/weights_e8_57.8358.pth
/content/drive/MyDrive/cvdisaster/convnext_base.fb_in22k_ft_in1k_384/212347/weights_e9_59.7015.pth
/content/drive/MyDrive/cvdisaster/convnext_base.fb_in22k_ft_in1k_384/212347/weights_e10_63.0597.pth
/content/drive/MyDrive/cvdisaster/convnext_base.fb_in22k_ft_in1k_384/212347/weights_end.pth


In [ ]:
import os, glob
from PIL import Image
!pip install pillow pillow-avif-plugin
import pillow_avif  # adds AVIF support to Pillow

SRC_DIR   = "/content"
PHOTO_DIR = "/content/my_photos"
SAT_DIR   = "/content/satellite_tiles"
EXTS      = (".png", ".jpg", ".jpeg", ".webp", ".avif", ".bmp", ".tif", ".tiff", ".heic")

os.makedirs(PHOTO_DIR, exist_ok=True)
os.makedirs(SAT_DIR, exist_ok=True)

files = [f for f in glob.glob(f"{SRC_DIR}/*") if os.path.isfile(f) and f.lower().endswith(EXTS)]
if not files:
    print("No images found in /content. Upload them to the top level of the file browser first.")

for f in files:
    name = os.path.basename(f)
    upper = name.upper()
    if "CROSSV" in upper:
        dest_dir = PHOTO_DIR
    elif "SATELLITEV" in upper:
        dest_dir = SAT_DIR
    else:
        print(f"SKIPPED {name}: name must contain CROSSV (ground photo) or SATELLITEV (aerial)")
        continue

    dest = os.path.join(dest_dir, os.path.splitext(name)[0] + ".png")
    try:
        with Image.open(f) as im:
            fmt = im.format
            im.convert("RGB").save(dest, "PNG")
    except Exception as e:
        print(f"FAILED  {name}: {e}")
        continue
    os.remove(f)
    print(f"OK      {name} ({fmt}) -> {dest}")

print("\nGround photos:", sorted(os.listdir(PHOTO_DIR)))
print("Aerial images:", sorted(os.listdir(SAT_DIR)))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 108.2 MB/s eta 0:00:00
SKIPPED LongBay_SATELLITE.jpg: name must contain CROSSV (ground photo) or SATELLITEV (aerial)
SKIPPED Hard_negative.jpg: name must contain CROSSV (ground photo) or SATELLITEV (aerial)
SKIPPED random_negative.png: name must contain CROSSV (ground photo) or SATELLITEV (aerial)
OK      GABRIELLE_HAWKESBAY_CROSSV.png (AVIF) -> /content/my_photos/GABRIELLE_HAWKESBAY_CROSSV.png
OK      GABRIELLE_HAWKESBAY_SATELLITEV2.png (PNG) -> /content/satellite_tiles/GABRIELLE_HAWKESBAY_SATELLITEV2.png
OK      GABRIELLE_HAWKESBAY_SATELLITEV4.png (PNG) -> /content/satellite_tiles/GABRIELLE_HAWKESBAY_SATELLITEV4.png
OK      GABRIELLE_HAWKESBAY_SATELLITEV.png (PNG) -> /content/satellite_tiles/GABRIELLE_HAWKESBAY_SATELLITEV.png
SKIPPED RodneyWard_SATELLITE.jpg: name must contain CROSSV (ground photo) or SATELLITEV (aerial)
SKIPPED 104078525924854.png: name must contain CROSSV (ground photo) or SATELLITEV (aerial)
SKIPPED Easy_negat

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/CVDisaster'):
    !git clone -q https://github.com/tum-bgd/CVDisaster.git /content/CVDisaster
%cd /content/CVDisaster/geolocalization

os.makedirs('/content/my_photos', exist_ok=True)
os.makedirs('/content/satellite_tiles', exist_ok=True)
print("Ready. Upload your photo(s) to /content/my_photos and the aerial to /content/satellite_tiles")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/CVDisaster/geolocalization
Ready. Upload your photo(s) to /content/my_photos and the aerial to /content/satellite_tiles


In [ ]:
# ============================================================
# Test Sample4Geo (fine-tuned on CVIAN): each ground photo vs ITS OWN satellite image only
# Pairing is by file name, e.g. ..._CROSSV2.png  <->  ..._SATELLITEV2.png
#                               ..._CROSS.PNG    <->  ..._SATELLITE.png
# ============================================================
import sys, csv, glob, os, re, json
from pathlib import Path
import cv2, numpy as np, torch, torch.nn.functional as F

# ---------------- SETTINGS (edit these) ----------------
REPO        = "/content/CVDisaster/geolocalization"
RUN_DIR     = max(glob.glob("/content/drive/MyDrive/cvdisaster/convnext_base.fb_in22k_ft_in1k_384/*/"),
                  key=os.path.getmtime)          # most recent training run
CHECKPOINT  = os.path.join(RUN_DIR, "weights_end.pth")
# The model retrained on left/right photos (from Sample4Geo_CVIAN_Perspective_Retrain.ipynb).
# If this folder has a trained model in it, that model is tested and photos are fed in directly.
# If not (or set to None), the panorama model above is tested, with MODE below.
PERSPECTIVE_RUN = None

# VERIFICATION: a photo is accepted (LIKELY MATCH) if its similarity to its satellite image is at least
# MATCH_THRESHOLD, otherwise CANNOT VERIFY. For the retrained model the threshold is read automatically from
# verification_threshold.json (set from the CVIAN test set in the training notebook, Step 9).
MATCH_THRESHOLD = 0.5   # used for the panorama model, or if the retrained model has no saved threshold yet
QUERIES     = "/content/my_photos"          # folder of ground photos
GALLERY     = "/content/satellite_tiles"    # folder of satellite images
OUT_CSV     = "/content/results.csv"

MODE         = "pad"    # panorama model only: "pad" = place photo in blank panorama, "stretch" = resize to 140x768
FOV          = 70.0     # horizontal field of view of the photos (degrees)
HEADING      = None     # compass heading in degrees if known, else None to sweep
HEADING_STEP = 30.0     # sweep step when HEADING is None
NORTH_COL    = 0.5      # where north sits in the training panoramas (fraction of width); only used if HEADING is set
BATCH_SIZE   = 16
# -------------------------------------------------------

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
GROUND_H, GROUND_W, SAT_SIZE = 140, 768, 384
MODEL_NAME = "convnext_base.fb_in22k_ft_in1k_384"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if PERSPECTIVE_RUN and os.path.exists(os.path.join(PERSPECTIVE_RUN, "config.json")) \
        and os.path.exists(os.path.join(PERSPECTIVE_RUN, "weights_best.pth")):
    with open(os.path.join(PERSPECTIVE_RUN, "config.json")) as f:
        GROUND_H, GROUND_W = json.load(f)["ground_size"]
    CHECKPOINT, MODE = os.path.join(PERSPECTIVE_RUN, "weights_best.pth"), "perspective"
    thr_file = os.path.join(PERSPECTIVE_RUN, "verification_threshold.json")
    if os.path.exists(thr_file):
        with open(thr_file) as f:
            thr = json.load(f)
        MATCH_THRESHOLD, THRESHOLD_NOTE = thr["threshold"], thr["explanation"]
        SAT_ROTATIONS = thr.get("satellite_rotations", "single")
    print("Testing the RETRAINED left/right model (photos fed in directly)")
else:
    print("Testing the PANORAMA model" + (f" (no retrained model in {PERSPECTIVE_RUN} yet)" if PERSPECTIVE_RUN else ""))
THRESHOLD_NOTE = globals().get("THRESHOLD_NOTE") if MODE == "perspective" else None
# how the satellite image is scored: "single" (as is), "mean4"/"max4" (average/best over 0, 90, 180, 270 degrees);
# chosen by Step 9 of the training notebook for the retrained model, "single" for the panorama model
SAT_ROTATIONS = globals().get("SAT_ROTATIONS", "single") if MODE == "perspective" else "single"
THRESHOLD_NOTE = THRESHOLD_NOTE or f"Threshold {MATCH_THRESHOLD} was set by hand."
print("Device:", device)
print("Checkpoint:", CHECKPOINT)

def list_images(path):
    path = Path(path)
    if path.is_file():
        return [path]
    imgs = sorted(p for p in path.iterdir() if p.suffix.lower() in IMG_EXTS)
    assert imgs, f"No images found in {path}"
    return imgs

def read_rgb(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    assert img is not None, f"Could not read {path}"
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def normalise(img):
    x = (img.astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1).copy())

def satellite_tensor(path):
    return normalise(cv2.resize(read_rgb(path), (SAT_SIZE, SAT_SIZE), interpolation=cv2.INTER_AREA))

def photo_strip(img, fov):
    w = max(1, round(GROUND_W * fov / 360.0))
    h = max(1, round(w * img.shape[0] / img.shape[1]))
    x = normalise(cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA))
    if h > GROUND_H:
        top = (h - GROUND_H) // 2
        x = x[:, top:top + GROUND_H]
    return x

def place_on_canvas(strip, centre_col):
    canvas = torch.zeros(3, GROUND_H, GROUND_W)   # 0 = mean colour after normalisation
    h, w = strip.shape[1:]
    top = (GROUND_H - h) // 2
    canvas[:, top:top + h, :w] = strip
    shift = int(round(centre_col - w / 2)) % GROUND_W
    return torch.roll(canvas, shifts=shift, dims=2)

def query_variants(img):
    if MODE == "perspective":   # retrained model: centre-crop the photo to the training shape, then resize
        h, w = img.shape[:2]
        if h / w > GROUND_H / GROUND_W:
            ch = round(w * GROUND_H / GROUND_W); img = img[(h - ch) // 2:(h - ch) // 2 + ch]
        else:
            cw = round(h * GROUND_W / GROUND_H); img = img[:, (w - cw) // 2:(w - cw) // 2 + cw]
        return [normalise(cv2.resize(img, (GROUND_W, GROUND_H), interpolation=cv2.INTER_AREA))], [None]
    if MODE == "stretch":
        return [normalise(cv2.resize(img, (GROUND_W, GROUND_H), interpolation=cv2.INTER_AREA))], [None]
    strip = photo_strip(img, FOV)
    headings = [HEADING] if HEADING is not None else [float(h) for h in np.arange(0, 360, HEADING_STEP)]
    return [place_on_canvas(strip, (NORTH_COL + h / 360.0) * GROUND_W) for h in headings], headings

# ---------------- pair each photo with its own satellite image ----------------
# The CROSSV / SATELLITEV (or CROSS / SATELLITE) tag and the file extension are removed,
# so GABRIELLE_HAWKESBAY_CROSSV2.png and GABRIELLE_HAWKESBAY_SATELLITEV2.png both become
# GABRIELLE_HAWKESBAY_#2. The tag only counts as a whole word, so e.g. CROSSING is left alone.
TAG = re.compile(r"(?<![A-Z])(SATELLITEV|SATELLITE|CROSSV|CROSS)(?=[^A-Z]|$)")

def match_key(path):
    return TAG.sub("#", Path(path).stem.upper(), count=1)

photos = list_images(QUERIES)
tiles = {}
for t in list_images(GALLERY):
    k = match_key(t)
    if k in tiles:
        print(f"WARNING: {t.name} and {tiles[k].name} both match the same photo name; using {tiles[k].name}")
        continue
    tiles[k] = t

pairs, used = [], set()
for q in photos:
    t = tiles.get(match_key(q))
    if t is None:
        print(f"SKIPPED {q.name}: no satellite image with a matching name in {GALLERY}")
        continue
    pairs.append((q, t))
    used.add(t)
for t in tiles.values():
    if t not in used:
        print(f"UNUSED  {t.name}: no ground photo with a matching name in {QUERIES}")
assert pairs, "No photo/satellite pairs found - check the file names"
print(f"\nPairs: {len(pairs)}")
for q, t in pairs:
    print(f"  {q.name}  <->  {t.name}")

# ---------------- load model ----------------
sys.path.insert(0, REPO)
from sample4geo.model import TimmModel
model = TimmModel(MODEL_NAME, pretrained=False, img_size=SAT_SIZE)
state = torch.load(CHECKPOINT, map_location="cpu")
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]
state = {k.removeprefix("module."): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
print(f"\nCheckpoint loaded: {len(missing)} missing, {len(unexpected)} unexpected keys"
      + ("  <-- WARNING: weights may not have loaded properly" if missing or unexpected else ""))
model = model.to(device).eval()

@torch.no_grad()
def embed(tensors):
    feats = []
    for i in range(0, len(tensors), BATCH_SIZE):
        x = torch.stack(tensors[i:i + BATCH_SIZE]).to(device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            f = model(x)
        feats.append(F.normalize(f.float(), dim=-1).cpu())
    return torch.cat(feats)

# ---------------- score each photo against its own satellite image ----------------
rows, VERDICTS = [], {}
print(f"\n{'Ground photo':<42}{'Satellite image':<44}{'Score':>8}{'Heading':>9}  Verdict")
for q, t in pairs:
    sat = satellite_tensor(t)
    ks = [0] if SAT_ROTATIONS == "single" else [0, 1, 2, 3]
    sat_feats = embed([torch.rot90(sat, k, dims=(1, 2)) for k in ks])   # one per satellite rotation
    variants, headings = query_variants(read_rgb(q))
    sims = embed(variants) @ sat_feats.T                       # headings tried x satellite rotations
    per_rot, best_v = sims.max(dim=0)                          # best heading for each rotation
    score = (per_rot.mean() if SAT_ROTATIONS == "mean4" else per_rot.max()).item()
    hd = headings[int(best_v[int(per_rot.argmax())])]
    verified = score >= MATCH_THRESHOLD
    VERDICTS[q.name] = {"score": score, "verified": verified}
    print(f"{q.name:<42}{t.name:<44}{score:>8.4f}"
          + (f"{hd:>8.0f}°" if hd is not None else f"{'-':>9}")
          + ("  LIKELY MATCH" if verified else "  CANNOT VERIFY"))
    rows.append({
        "query": q.name, "satellite": t.name,
        "score": f"{score:.6f}",
        "best_heading": "" if hd is None else f"{hd:.0f}",
        "verdict": "LIKELY MATCH" if verified else "CANNOT VERIFY",
        "threshold": MATCH_THRESHOLD,
    })
print(f"\nLIKELY MATCH = similarity of at least {MATCH_THRESHOLD:.3f}. {THRESHOLD_NOTE}")
print(f"Satellite scoring: {SAT_ROTATIONS}")

with open(OUT_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
print(f"\nSaved results to {OUT_CSV}")

ValueError: max() iterable argument is empty

In [ ]:
import cv2, numpy as np, torch, matplotlib.pyplot as plt, glob, os

# ---------------- SETTINGS ----------------
PHOTO_DIR = "/content/my_photos"
SAT_DIR   = "/content/satellite_tiles"
TILE      = 384        # square size in aerial pixels (auto-shrinks if an aerial is smaller)
STRIDE    = 48         # step between squares; smaller = finer heatmap
TRUE_XY   = {}         # optional, per pair, e.g. {"GABRIELLE_HAWKESBAY_2": (170, 270)}
M_PER_PX  = None       # metres per aerial pixel, if known

# The verdict (LIKELY MATCH / CANNOT VERIFY) comes from cell 14: the photo's similarity to its satellite
# image against MATCH_THRESHOLD. This cell adds a heatmap showing which part of the satellite image matches best;
# a CANNOT VERIFY photo is stamped and gets no "best match" mark.
# ------------------------------------------

# Pairing reuses cell 14 (TAG, match_key, list_images), so run cell 14 first.
# It accepts .png and .PNG, and pairs CROSSV<->SATELLITEV as well as CROSS<->SATELLITE (the false test).
def pair_label(path):
    stem = Path(path).stem
    m = TAG.search(stem.upper())
    if not m:
        return stem
    before, after = stem[:m.start()].rstrip("_- "), stem[m.end():].lstrip("_- ")
    return (before + ("_" + after if after else "")) or stem

def find_pairs():
    sats = {match_key(s): s for s in list_images(SAT_DIR)}
    pairs = []
    for p in list_images(PHOTO_DIR):
        if match_key(p) in sats:
            pairs.append((pair_label(p), str(p), str(sats[match_key(p)])))
        else:
            print(f"WARNING: no matching satellite image for {p.name}")
    return pairs

def heatmap_for(label, photo_path, aerial_path, true_xy=None):
    big = read_rgb(aerial_path)
    H, W = big.shape[:2]
    t = min(TILE, H, W)                                  # squares always fully inside the image
    xs = list(range(0, W - t + 1, STRIDE))
    ys = list(range(0, H - t + 1, STRIDE))
    coords  = [(x, y) for y in ys for x in xs]
    centres = [(x + t // 2, y + t // 2) for x, y in coords]

    feats = []
    for i in range(0, len(coords), BATCH_SIZE):
        chunk = [normalise(cv2.resize(big[y:y + t, x:x + t], (SAT_SIZE, SAT_SIZE),
                                      interpolation=cv2.INTER_AREA))
                 for x, y in coords[i:i + BATCH_SIZE]]
        feats.append(embed(chunk))
    feats = torch.cat(feats)

    photo = read_rgb(photo_path)
    variants, headings = query_variants(photo)
    sims = embed(variants) @ feats.T
    best, best_v = sims.max(dim=0)
    best = best.numpy()
    order = np.argsort(-best)
    b = order[0]
    bx, by = centres[b]

    res = {"pair": label, "best_centre": (bx, by), "best_score": best[b],
           "spread": best[b] - np.median(best), "error": None, "true_rank": None,
           "squares": len(centres), "tile": t}
    if true_xy is not None:
        d = np.hypot(*(np.array(centres) - np.array(true_xy)).T)
        res["true_rank"] = int(np.where(order == int(d.argmin()))[0][0]) + 1
        res["error"] = float(np.hypot(bx - true_xy[0], by - true_xy[1]))

    # ---- verdict: is this clearly a match, or can it not be verified? ----
    score = float(best[b])
    v = VERDICTS.get(os.path.basename(photo_path))      # decided in cell 14
    res["verified"] = bool(v and v["verified"])
    if res["verified"]:
        verdict, colour = "LIKELY MATCH", "#1a7f37"
        reason = (f"Similarity to this satellite image {v['score']:.3f} is at least the {MATCH_THRESHOLD:.3f} needed. "
                  f"The cross marks the best-matching area.")
    else:
        verdict, colour = "CANNOT VERIFY", "#c62828"
        reason = ("This photo could not be verified against this satellite image: " +
                  (f"its similarity {v['score']:.3f} is below the {MATCH_THRESHOLD:.3f} needed."
                   if v else "it was not scored in cell 14 - run cell 14 first."))
    res["verdict"], res["reason"] = verdict, reason

    fig, (a0, a1) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 1.4]})
    a0.imshow(photo); a0.set_title(f"{label} – ground photo"); a0.axis("off")
    a1.imshow(big)
    im = a1.imshow(best.reshape(len(ys), len(xs)), cmap="inferno", alpha=0.5,
                   extent=(xs[0] + t/2 - STRIDE/2, xs[-1] + t/2 + STRIDE/2,
                           ys[-1] + t/2 + STRIDE/2, ys[0] + t/2 - STRIDE/2),
                   interpolation="bilinear")
    a1.set_xlim(0, W); a1.set_ylim(H, 0); a1.axis("off")
    fig.colorbar(im, ax=a1, fraction=0.03, label="similarity")
    if res["verified"]:
        a1.scatter([bx], [by], marker="x", s=200, c="cyan", label="best match")
    else:   # disclaimer stamped on the satellite image itself
        a1.text(0.5, 0.5, "CANNOT VERIFY", transform=a1.transAxes, ha="center", va="center",
                rotation=20, fontsize=38, fontweight="bold", color=colour,
                bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=colour,
                          linewidth=4, alpha=0.55))
    if true_xy is not None:
        a1.scatter([true_xy[0]], [true_xy[1]], marker="*", s=300, c="lime", label="true position")
    if a1.get_legend_handles_labels()[0]:
        a1.legend(loc="upper right")
    if res["verified"]:
        title = f"Best-matching area {score:.3f} at {(bx, by)} ({len(centres)} squares of {t}px)"
    else:
        title = f"Highest area score {score:.3f}, not a confirmed match ({len(centres)} squares of {t}px)"
    if res["error"] is not None:
        title += f" | {res['error']:.0f} px from truth" + (f" (~{res['error'] * M_PER_PX:.0f} m)" if M_PER_PX else "")
    a1.set_title(title)
    plt.tight_layout(rect=(0, 0, 1, 0.86))
    fig.text(0.5, 0.975, verdict, ha="center", va="top", fontsize=17, fontweight="bold", color="white",
             bbox=dict(boxstyle="round,pad=0.4", facecolor=colour, edgecolor="none"))
    fig.text(0.5, 0.905, reason, ha="center", va="top", fontsize=11, color=colour, wrap=True)
    plt.show()
    return res

pairs = find_pairs()
print(f"Found {len(pairs)} pair(s): {[p[0] for p in pairs]}\n")
results = [heatmap_for(label, ph, sat, TRUE_XY.get(label)) for label, ph, sat in pairs]

print(f"\n{'Pair':<32}{'Verdict':<16}{'Best score':>11}{'Spread':>9}{'Best centre':>16}{'Error px':>10}{'True rank':>11}")
for r in results:
    err = f"{r['error']:.0f}" if r["error"] is not None else "-"
    rk  = f"{r['true_rank']}/{r['squares']}" if r["true_rank"] else "-"
    print(f"{r['pair']:<32}{r['verdict']:<16}{r['best_score']:>11.3f}{r['spread']:>9.3f}{str(r['best_centre']):>16}{err:>10}{rk:>11}")
print(f"\nVerdict from cell 14: LIKELY MATCH = similarity of at least {MATCH_THRESHOLD:.3f}. {THRESHOLD_NOTE}")